In [36]:
import pandas as pd

#Improting all the CSV and combining them into one dataframe 
df = pd.concat([
    pd.read_csv("/Users/tarlinggorgun/Documents/semseter_3/pixie/pixie_1.csv"),
    pd.read_csv("/Users/tarlinggorgun/Documents/semseter_3/pixie/pixie_2.csv"),
    pd.read_csv("/Users/tarlinggorgun/Documents/semseter_3/pixie/pixie_3.csv"),
    pd.read_csv("/Users/tarlinggorgun/Documents/semseter_3/pixie/pixie_4.csv"),
    pd.read_csv("/Users/tarlinggorgun/Documents/semseter_3/pixie/pixie_5.csv")
], ignore_index=True)

print(df)

#Turning the timestamps into datatime values
df["@timestamp"] = pd.to_datetime(
    df["@timestamp"],
    format="%b %d, %Y @ %H:%M:%S.%f"
)

#From event column only keep game start and game end 
games = df[df["event"].isin(["GAME_START", "GAME_END"])].copy()

#creates a new dataframe with only the game start 
starts = games[games["event"] == "GAME_START"].copy()

#Creates a new dataframe with only the game end 
ends = games[games["event"] == "GAME_END"].copy()

#Rename the timestamp column to game start and generic string to game name in the start dataframe 
starts = starts.rename(columns={
    "@timestamp": "game_start",
    "generic_string": "game_name"
})

#rename the timestamp column to game end and generic string to game name in the end dataframe 
ends = ends.rename(columns={
    "@timestamp": "game_end",
    "generic_string": "game_name"
})

#Sort the hostname to have the corresponding game name and game start 
starts = starts.sort_values(["hostname", "game_name", "game_start"])

#creates a new column called next_game_start to store when does the same game start again on the same pixie 
starts["next_game_start"] = (

    #groups the hostname and game name and looks at game start
    starts.groupby(["hostname", "game_name"])["game_start"]

    #Moves the start time one row up so each start time can be compared to the next start time for the same game on the same pixie
    .shift(-1)
)

#create a new dataframe called session to match game start and game end to correct game name and hostname 
sessions = pd.merge_asof( #matches rows based on date and time 
    
    #sorts the start dataframe with the game start 
    starts.sort_values("game_start"),

    #Sorts the end data frame with game end
    ends.sort_values("game_end"),

    #Makes it so the game start is the starting point to find the match 
    left_on="game_start",

    #Looks for the matching game end that corresponds to the game start 
    right_on="game_end",

    #matches the game start and end to the correct hostname and game name, only when hostanme and game name are same 
    by=["hostname", "game_name"],

    #Looks for the game end time after the game start time 
    direction="forward"
)

#It fixes the issue from previous step where the game end time is not the correct game end for the game start 
sessions.loc[ #find specific row and a specific column 
    
    #Looks if there is another game start, check in the game_next column 
    sessions["next_game_start"].notna() &

    #Is the matched game end happening at or after the next game start for the same game, if true then game end was matched wrongly 
    (sessions["game_end"] >= sessions["next_game_start"]),

    #only chnage game end 
    "game_end"

    #if any missing value replace with NaT
] = pd.NaT

#removes the next game start column from the dataframe
sessions = sessions.drop(columns=["next_game_start"])

#maps out the location for each pixie based on the hostname 
location_map = {
    "p-03cbbcb4f4d3": "Zurich, Switzerland, Europe",
    "p-03cbbcb7d615": "Melbourne, Australia, Oceania",
    "p-03cbbd5b87c0": "France, Europe",
    "p-03cbbd5b9347": "France, Europe",
    "p-03cbbd5bd5b3": "Switzerland, Europe"
}

#For each hostname look for the correct location in the location map 
sessions["location"] = sessions["hostname"].map(location_map)

#find the total minutes played for each game session and rounds it up to 2 decimal places 
sessions["total_minutes"] = (
    (sessions["game_end"] - sessions["game_start"])
    .dt.total_seconds() / 60
).round(2)

#keeps only the column names wanted for the final dataframe 
sessions = sessions[
    ["hostname",
     "game_name",
     "game_start",
     "game_end",
     "total_minutes",
     "location"]
]


                        @timestamp              event        hostname  \
0      Jul 24, 2026 @ 16:23:18.014           SHUTDOWN  p-03cbbcb4f4d3   
1      Jul 24, 2026 @ 16:23:07.180           GAME_END  p-03cbbcb4f4d3   
2      Jul 24, 2026 @ 16:17:38.762         GAME_START  p-03cbbcb4f4d3   
3      Jul 24, 2026 @ 16:17:35.960  CALIBRATION_START  p-03cbbcb4f4d3   
4      Jul 24, 2026 @ 16:17:32.551           GAME_END  p-03cbbcb4f4d3   
...                            ...                ...             ...   
15601   Mar 4, 2025 @ 17:51:41.944  CALIBRATION_START  p-03cbbcb7d615   
15602   Mar 4, 2025 @ 17:51:41.396            STARTUP  p-03cbbcb7d615   
15603   Mar 4, 2025 @ 17:51:23.748           SHUTDOWN  p-03cbbcb7d615   
15604   Mar 4, 2025 @ 17:47:00.031  CALIBRATION_START  p-03cbbcb7d615   
15605   Mar 4, 2025 @ 17:46:58.586            STARTUP  p-03cbbcb7d615   

       generic_string generic_float geoip.city_name geoip.continent_name  \
0             v18.6.0           0.0            

In [37]:
import pandas as pd

#load new csv 
raw = pd.read_csv(
    "pixie/pixie_events_raw.csv",
    low_memory=False
)

#gets and makes a copy of game start form the event name 
raw_starts = raw[
    raw["event_name"] == "GAME_START"
].copy()

#Turns the timestamp into datatime values for total time played 
raw_starts["timestamp_utc"] = pd.to_datetime(
    raw_starts["timestamp_utc"],
    utc=True
)

#renames the timestamp to game start and generic string to game name
raw_starts = raw_starts.rename(columns={
    "timestamp_utc": "game_start",
    "generic_string": "game_name"
})


# Keep the information we want from GAME_START
raw_starts = raw_starts[
    [
        "hostname",
        "game_start",
        "game_name",
        "generic_float",
        "serial",
        "serial_tover_view",
        "session_uuid",
        "app_uuid",
        "n_copies"
    ]
]


# Sort by Pixie, game name and game start time
raw_starts = raw_starts.sort_values(
    ["hostname", "game_name", "game_start"]
)

#find the game next to prevent one game end from being inccorectly used
#creates a new column called next_game_start to store when does the same game start again on the same pixie 
raw_starts["next_game_start"] = (
    raw_starts
    .groupby(["hostname", "game_name"])["game_start"]  #groups the hostname and game name and looks at game start
    
    #Moves the start time one row up so each start time can be compared to the next start time for the same game on the same pixie
    .shift(-1)
)


#gets the game end from the event name column and makes a copy 
raw_ends = raw[
    raw["event_name"] == "GAME_END"
].copy()

#Truns the time stamp into the dataframe value fro the total time played later 
raw_ends["timestamp_utc"] = pd.to_datetime(
    raw_ends["timestamp_utc"],
    utc=True
)

#Renames the timestamp to game end and generic string to game name
raw_ends = raw_ends.rename(columns={
    "timestamp_utc": "game_end",
    "generic_string": "game_name"
})


# Keep only the information needed to match game end to the correct game name and hostname 
raw_ends = raw_ends[
    [
        "hostname",
        "game_end",
        "game_name"
    ]
]


# Sort by time
raw_ends = raw_ends.sort_values(
    ["hostname", "game_name", "game_end"]
)


#create a new dataframe called all_sessions to match game start and game end to correct game name and hostname 
all_sessions = pd.merge_asof(

    #sorts the start dataframe with the game start 
    raw_starts.sort_values("game_start"),

    #Sorts the end data frame with game end
    raw_ends.sort_values("game_end"),

    #Makes it so the game start is the starting point to find the match 
    left_on="game_start",

    #Looks for the matching game end that corresponds to the game start 
    right_on="game_end",

    #matches the game start and end to the correct hostname and game name, only when hostanme and game name are same 
    by=["hostname", "game_name"],

    #Looks for the game end time after the game start time 
    direction="forward"
)


# If another GAME_START happens before the GAME_END,
# the earlier session does not have a valid GAME_END.

all_sessions.loc[
    all_sessions["next_game_start"].notna()
    &
    (all_sessions["game_end"] >= all_sessions["next_game_start"]),
    "game_end"
] = pd.NaT


#calculates the total time played for each game by minusing the start time from the end time 
all_sessions["total_minutes"] = (
    (
        all_sessions["game_end"]
        - all_sessions["game_start"]
    )
    .dt.total_seconds()
    / 60 #dividing by 60 to find minutes 
).round(2) #round to 2 decimal places




#added a location map but there isnt any location information in the new csv its so when we do get i cant add easily 
all_sessions["location"] = (
    all_sessions["hostname"].map(location_map)
)


#remove the game next column 
all_sessions = all_sessions.drop(
    columns=["next_game_start"]
)



#keep only the columns wanted for the final dataframe 
all_sessions = all_sessions[
    [
        "hostname",
        "game_name",
        "game_start",
        "game_end",
        "total_minutes",
        "location",
        "generic_float",
        "serial",
        "serial_tover_view",
        "session_uuid",
        "app_uuid",
        "n_copies"
    ]
]




In [39]:
#combine the old and new pixie dataframe 

# Load the old final dataframe
old_sessions = pd.read_csv(
    "/Users/tarlinggorgun/Documents/semseter_3/pixie/pixie_games_final.csv"
)

# Convert timestamps to datatime values for game start 
old_sessions["game_start"] = pd.to_datetime(
    old_sessions["game_start"],
    utc=True
)

#convert timestamps for datatime values for game end 
old_sessions["game_end"] = pd.to_datetime(
    old_sessions["game_end"],
    utc=True
)


#because some of the old pixie data is in the new it removes duplicates 
old_hosts = old_sessions["hostname"].unique()



#add the new columns in that didnt alreayd exist in the first datafram 

for col in [
    "generic_float",
    "serial",
    "serial_tover_view",
    "session_uuid",
    "app_uuid",
    "n_copies"
]:
    old_sessions[col] = pd.NA #the old csv didnt have these columns so the missing value is just NaN


#which columns to keep and in this order 
old_sessions = old_sessions[
    [
        "hostname",
        "game_name",
        "game_start",
        "game_end",
        "total_minutes",
        "location",
        "generic_float",
        "serial",
        "serial_tover_view",
        "session_uuid",
        "app_uuid",
        "n_copies"
    ]
]


#from the new raw data only keep the pixie that are not already shown in the old data 
new_pixie_sessions = all_sessions[
    ~all_sessions["hostname"].isin(old_hosts)
].copy()



#combine the old and new pixie dataframe into one called sessions 
sessions = pd.concat(
    [
        old_sessions,
        new_pixie_sessions
    ],
    ignore_index=True
)


#remove any duplicate rows 
sessions = sessions.drop_duplicates().reset_index(drop=True)


#sort them chrnologically
sessions = sessions.sort_values(
    "game_start"
).reset_index(drop=True)




/var/folders/pl/trdh5z2d22zgn2cz37c5m_d00000gn/T/ipykernel_98957/215898395.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sessions = pd.concat(


In [35]:
sessions.to_csv(
    "pixie_games_combined_final.csv",
    index=False
)